# 项目说明
##### 该文件旨在记录 项目诞生的过程

### 阶段0 docker 搭建
apt-get update  
apt-get install -y ca-certificates curl gnupg wget  

install -m 0755 -d /etc/apt/keyrings  
curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc  
chmod a+r /etc/apt/keyrings/docker.asc  

echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu $(. /etc/os-release && echo "$VERSION_CODENAME") stable" > /etc/apt/sources.list.d/docker.list  

apt-get update
apt-get install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin

docker --version
docker compose version
systemctl status docker --no-pager

docker run hello-world  
##### 若出现拉去失败 考虑加速器
mkdir -p /etc/docker
nano /etc/docker/daemon.json
{
  "registry-mirrors": [
    "https://docker.m.daocloud.io",
    "https://dockerproxy.com",
    "https://docker.1ms.run"
  ]
}
systemctl daemon-reload
systemctl restart docker

mkdir -p /root/milvus
cd /root/milvus
wget https://github.com/milvus-io/milvus/releases/download/v2.6.13/milvus-standalone-docker-compose.yml -O docker-compose.yml

##### 端口占用问题 修改端口
nano ~/milvus/docker-compose.yml
找到minio:  ports: 从“9000 ： 9000” 改成“19000：9000“ （前者是云服务器的端口 后者是docker需要的/容器端口）



### 阶段1 环境搭建
1. 上传raw data(as pdf)
2. 搭建虚拟环境 （python -m venv renv）
2.1 进入虚拟环境 （在文件：RAG_Agent 中， 输入：source renv/bin/activate）
3. 在虚拟环境中安装mineru 



pip install --upgrade pip     
pip install uv     
uv pip install -U "mineru[all]"  
若安装速度很慢 可使用国内的镜像（大幅提升速度） 
uv pip install -U "mineru[all]" -i https://mirrors.aliyun.com/pypi/simple/  
export MINERU_MODEL_SOURCE=modelscope 


### 阶段二 数据读入
#### 2.1 ingestion
##### 2.1.1 parsing(解析)
（Apr 3rd ） 
使用mineru 将 pdf格式转化为json格式  
例子： mineru -p data/raw/test0403/5.pdf -o data/mineru_out -b pipeline

对mineru_out 进行按页切分
python src/mineru/mineru_clean_page.py data/mineru_out/5/auto/5_content_list.json --output_dir data/clean_json

检查 token size.  
python src/mineru/check_page_tokens.py data/clean_json/1_content_ingestion.json 

构建 chunker.py (size = 400, overlap = 80)    
python src/mineru/chunker.py data/clean_json/4_content_ingestion.json    

使用 api-key 进行 embedding    
export SILICONFLOW_API_KEY="sk-qvjezwqyoqcpnxidmvztgbtehnusxuzbnubwtjoqlscxkwug"

使用match.py
python src/query/match.py "S6系列无人车产品有什么系统？" --top_k 3 



##### 优化
对于 mineru_out 的 type = table 为了生存大模型可读的版本 进行 结构性优化    
python src/mineru/convert_mineru_tables.py \
  --input data/mineru_out/4/auto/4_content_list.json \
  --output data/mineru_out/4/auto/4_content_list_table_llm.json

### 将图片保存至minio 且将路径修改为可访问的
文件：src/mineru/image_save.py  
请输入 json_path: ~/project/RAG_Agent/data/mineru_out/4/auto/4_content_list.json  
请输入 images_dir: ~/project/RAG_Agent/data/mineru_out/4/auto/images